# Los Cuatro Principios de la Computacion Autonomica Aplicados al Agendamiento de Reuniones

Este notebook demuestra como implementar los cuatro principios fundamentales de la computacion autonomica en el contexto de un asistente virtual para agendar reuniones.

La computacion autonomica es un paradigma inspirado en el sistema nervioso autonomo humano, que permite a los sistemas informaticos gestionar y adaptarse automaticamente a cambios en su entorno con minima intervencion humana.

## Los Cuatro Principios Fundamentales

### 1. Auto-configuracion (Self-Configuration)
El sistema debe configurarse automaticamente segun el contexto y las necesidades sin intervencion manual.
- Detecta el tipo de reunion basandose en el contexto
- Asigna duraciones apropiadas segun el tipo de evento
- Configura recordatorios automaticamente segun la importancia
- Ajusta parametros segun las preferencias del usuario

### 2. Auto-optimizacion (Self-Optimization)
El sistema debe monitorear y mejorar continuamente su rendimiento y eficiencia.
- Optimiza horarios evitando conflictos y sobrecarga
- Sugiere mejores horarios basandose en patrones historicos
- Aprende de decisiones previas para mejorar sugerencias futuras
- Balancea la carga de reuniones a lo largo del dia

### 3. Auto-curacion (Self-Healing)
El sistema debe detectar, diagnosticar y reparar problemas automaticamente.
- Detecta cuando faltan datos criticos para agendar una reunion
- Diagnostica que informacion especifica falta
- Repara solicitando los datos faltantes de forma inteligente
- Verifica que la reparacion fue exitosa antes de continuar

### 4. Auto-proteccion (Self-Protection)
El sistema debe defenderse de ataques maliciosos y usos indebidos.
- Valida todas las entradas del usuario para prevenir inyeccion de codigo
- Protege informacion sensible como tokens y credenciales
- Detecta patrones anomalos de uso
- Implementa rate limiting para prevenir abusos
- Sanitiza datos antes de procesarlos

In [ ]:
# Importaciones necesarias para el sistema autonomico
from openai import OpenAI
import re
from datetime import datetime, timedelta
from typing import Union, Dict, List, Optional
import json
import hashlib
import time

# Esto es parte del principio de AUTO-PROTECCION
client = OpenAI(api_key="tu_api_key_aqui")

print("Sistema autonomico de agendamiento de reuniones inicializado")

## Principio 1: AUTO-CONFIGURACION (Self-Configuration)

La auto-configuracion permite que el sistema se adapte automaticamente al contexto sin necesidad de configuracion manual. En el contexto de agendamiento de reuniones, esto significa:

1. Detectar automaticamente el tipo de reunion (informal, formal, estrategica, etc.)
2. Asignar duracion apropiada segun el tipo detectado
3. Configurar recordatorios segun la importancia
4. Adaptar el formato de la reunion segun los participantes

In [ ]:
# ========================================================================
# PRINCIPIO 1: AUTO-CONFIGURACION
# Sistema que detecta y configura automaticamente parametros de reunion
# ========================================================================

class AutoConfigurationEngine:
    """
    Motor de auto-configuracion que analiza el contexto de una reunion
    y configura automaticamente sus parametros sin intervencion manual.
    
    Este componente implementa el primer principio de la computacion autonomica:
    la capacidad del sistema de configurarse a si mismo segun las necesidades.
    """
    
    def __init__(self):
        # Configuraciones predeterminadas segun tipo de reunion
        # Estas reglas permiten que el sistema se auto-configure sin intervencion
        self.meeting_types_config = {
            'estrategica': {
                'duracion_minutos': 90,
                'recordatorios': [1440, 60, 15],  # 1 dia, 1 hora, 15 min antes
                'prioridad': 'alta',
                'requiere_agenda': True,
                'formato': 'formal'
            },
            'seguimiento': {
                'duracion_minutos': 30,
                'recordatorios': [60, 15],  # 1 hora, 15 min antes
                'prioridad': 'media',
                'requiere_agenda': False,
                'formato': 'semi-formal'
            },
            'informal': {
                'duracion_minutos': 20,
                'recordatorios': [15],  # 15 min antes
                'prioridad': 'baja',
                'requiere_agenda': False,
                'formato': 'casual'
            },
            'presentacion': {
                'duracion_minutos': 60,
                'recordatorios': [1440, 120, 30],  # 1 dia, 2 horas, 30 min antes
                'prioridad': 'alta',
                'requiere_agenda': True,
                'formato': 'formal'
            },
            'brainstorming': {
                'duracion_minutos': 45,
                'recordatorios': [60, 15],
                'prioridad': 'media',
                'requiere_agenda': False,
                'formato': 'creativo'
            }
        }
        
        # Palabras clave para detectar tipo de reunion automaticamente
        # Este es el mecanismo de deteccion automatica
        self.keywords = {
            'estrategica': ['estrategia', 'planificacion', 'objetivos', 'vision', 'direccion'],
            'seguimiento': ['seguimiento', 'status', 'avance', 'progreso', 'actualizacion'],
            'informal': ['cafe', 'charla', 'conversacion', 'informal', 'casual'],
            'presentacion': ['presentacion', 'demo', 'demostracion', 'pitch', 'exposicion'],
            'brainstorming': ['brainstorm', 'ideas', 'creativo', 'innovacion', 'lluvia']
        }
    
    def detect_meeting_type(self, description: str, asunto: str) -> str:
        """
        AUTO-CONFIGURACION: Detecta automaticamente el tipo de reunion
        analizando el asunto y la descripcion.
        
        Este metodo implementa la deteccion contextual automatica,
        un componente clave de la auto-configuracion.
        """
        text = f"{description} {asunto}".lower()
        
        # Contador de coincidencias por tipo
        scores = {meeting_type: 0 for meeting_type in self.keywords.keys()}
        
        # Analizar texto buscando palabras clave
        for meeting_type, keywords in self.keywords.items():
            for keyword in keywords:
                if keyword in text:
                    scores[meeting_type] += 1
        
        # Retornar el tipo con mayor puntuacion
        # Si no hay coincidencias, usar 'seguimiento' como predeterminado
        detected_type = max(scores, key=scores.get)
        if scores[detected_type] == 0:
            detected_type = 'seguimiento'  # Tipo predeterminado
        
        print(f"[AUTO-CONFIGURACION] Tipo de reunion detectado: {detected_type}")
        return detected_type
    
    def auto_configure(self, asunto: str, descripcion: str) -> Dict:
        """
        AUTO-CONFIGURACION: Configura automaticamente todos los parametros
        de la reunion basandose en el tipo detectado.
        
        Este es el metodo principal que orquesta la auto-configuracion completa.
        Demuestra como el sistema puede tomar decisiones inteligentes sin
        intervencion del usuario.
        """
        print("\n[AUTO-CONFIGURACION] Iniciando configuracion automatica...")
        
        # Paso 1: Detectar tipo de reunion
        meeting_type = self.detect_meeting_type(descripcion, asunto)
        
        # Paso 2: Obtener configuracion para este tipo
        config = self.meeting_types_config.get(meeting_type, self.meeting_types_config['seguimiento'])
        
        # Paso 3: Crear configuracion completa
        auto_config = {
            'tipo': meeting_type,
            'duracion_minutos': config['duracion_minutos'],
            'hora_fin_calculada': None,  # Se calculara con la hora de inicio
            'recordatorios_minutos_antes': config['recordatorios'],
            'prioridad': config['prioridad'],
            'requiere_agenda': config['requiere_agenda'],
            'formato': config['formato']
        }
        
        print(f"[AUTO-CONFIGURACION] Duracion asignada: {config['duracion_minutos']} minutos")
        print(f"[AUTO-CONFIGURACION] Prioridad: {config['prioridad']}")
        print(f"[AUTO-CONFIGURACION] Recordatorios: {config['recordatorios']} minutos antes")
        print(f"[AUTO-CONFIGURACION] Formato: {config['formato']}")
        print("[AUTO-CONFIGURACION] Configuracion automatica completada con exito\n")
        
        return auto_config

# Ejemplo de uso del motor de auto-configuracion
config_engine = AutoConfigurationEngine()

# Caso 1: Reunion estrategica
print("=" * 70)
print("EJEMPLO 1: Reunion de planificacion estrategica")
print("=" * 70)
config1 = config_engine.auto_configure(
    asunto="Planificacion estrategica Q2",
    descripcion="Definir objetivos y estrategia para el segundo trimestre"
)

# Caso 2: Cafe informal
print("\n" + "=" * 70)
print("EJEMPLO 2: Cafe informal")
print("=" * 70)
config2 = config_engine.auto_configure(
    asunto="Cafe con Maria",
    descripcion="Charla informal para conocernos mejor"
)

## Principio 2: AUTO-OPTIMIZACION (Self-Optimization)

La auto-optimizacion permite que el sistema monitoree su rendimiento y mejore continuamente. En el contexto de agendamiento de reuniones:

1. Optimiza horarios evitando conflictos y sobrecarga de agenda
2. Sugiere mejores horarios basandose en patrones de productividad
3. Balancea la carga de reuniones a lo largo del dia
4. Aprende de decisiones previas para mejorar recomendaciones futuras

In [ ]:
# ========================================================================
# PRINCIPIO 2: AUTO-OPTIMIZACION
# Sistema que optimiza automaticamente los horarios de reuniones
# ========================================================================

class AutoOptimizationEngine:
    """
    Motor de auto-optimizacion que analiza patrones, evita conflictos
    y sugiere los mejores horarios para reuniones.
    
    Este componente implementa el segundo principio de computacion autonomica:
    la capacidad del sistema de mejorar continuamente su rendimiento
    y eficiencia sin intervencion manual.
    """
    
    def __init__(self):
        # Simular agenda existente (en produccion, esto vendria de una base de datos)
        self.agenda = []
        
        # Patrones de productividad basados en investigacion
        # Horas con mayor energia cognitiva
        self.peak_productivity_hours = [9, 10, 11, 15, 16]
        
        # Limites para evitar sobrecarga
        self.max_meetings_per_day = 6
        self.max_consecutive_meetings = 3
        self.min_break_minutes = 15  # Tiempo minimo entre reuniones
    
    def add_to_agenda(self, meeting: Dict):
        """
        Agrega una reunion a la agenda para analisis de optimizacion.
        """
        self.agenda.append(meeting)
    
    def check_conflicts(self, fecha: str, hora_inicio: str, duracion_minutos: int) -> List[Dict]:
        """
        AUTO-OPTIMIZACION: Detecta conflictos con reuniones existentes.
        
        Este metodo previene sobreposiciones, un aspecto clave de la optimizacion.
        """
        conflicts = []
        
        # Convertir hora de inicio a datetime
        inicio_dt = datetime.strptime(f"{fecha} {hora_inicio}", "%Y-%m-%d %H:%M")
        fin_dt = inicio_dt + timedelta(minutes=duracion_minutos)
        
        # Buscar conflictos en la agenda
        for meeting in self.agenda:
            if meeting['fecha'] != fecha:
                continue
            
            meeting_inicio = datetime.strptime(f"{meeting['fecha']} {meeting['hora']}", "%Y-%m-%d %H:%M")
            meeting_fin = meeting_inicio + timedelta(minutes=meeting.get('duracion', 30))
            
            # Verificar sobreposicion
            if (inicio_dt < meeting_fin) and (fin_dt > meeting_inicio):
                conflicts.append(meeting)
        
        return conflicts
    
    def calculate_load_score(self, fecha: str, hora: str) -> float:
        """
        AUTO-OPTIMIZACION: Calcula un score de carga para un horario.
        
        Un score mas bajo indica un horario mas optimo.
        Considera:
        - Numero de reuniones ese dia
        - Distancia con otras reuniones
        - Horarios de alta productividad
        """
        score = 0.0
        hora_int = int(hora.split(':')[0])
        
        # Contar reuniones del mismo dia
        meetings_same_day = [m for m in self.agenda if m['fecha'] == fecha]
        score += len(meetings_same_day) * 10  # Penalizar dias con muchas reuniones
        
        # Premiar horas de alta productividad
        if hora_int in self.peak_productivity_hours:
            score -= 20  # Bonus por hora productiva
        
        # Penalizar horas muy tempranas o muy tardias
        if hora_int < 8 or hora_int > 18:
            score += 30
        
        # Penalizar hora de almuerzo (12-14)
        if 12 <= hora_int <= 13:
            score += 15
        
        return score
    
    def suggest_optimal_times(self, fecha: str, duracion_minutos: int, cantidad: int = 3) -> List[Dict]:
        """
        AUTO-OPTIMIZACION: Sugiere los mejores horarios disponibles.
        
        Este metodo analiza multiples factores y retorna las opciones mas optimas:
        - Sin conflictos
        - En horarios productivos
        - Con distribucion balanceada
        - Con descansos apropiados
        """
        print(f"\n[AUTO-OPTIMIZACION] Analizando mejores horarios para {fecha}...")
        
        suggestions = []
        
        # Generar candidatos (horarios de 8 AM a 6 PM cada 30 minutos)
        for hour in range(8, 19):
            for minute in [0, 30]:
                if hour == 18 and minute == 30:
                    break  # No agendar tan tarde
                
                hora_str = f"{hour:02d}:{minute:02d}"
                
                # Verificar conflictos
                conflicts = self.check_conflicts(fecha, hora_str, duracion_minutos)
                
                if not conflicts:
                    # Calcular score de optimizacion
                    score = self.calculate_load_score(fecha, hora_str)
                    
                    suggestions.append({
                        'hora': hora_str,
                        'score': score,
                        'razon': self._get_score_explanation(score, hora_str)
                    })
        
        # Ordenar por score (menor es mejor) y retornar top N
        suggestions.sort(key=lambda x: x['score'])
        top_suggestions = suggestions[:cantidad]
        
        print(f"[AUTO-OPTIMIZACION] Se encontraron {len(suggestions)} horarios disponibles")
        print(f"[AUTO-OPTIMIZACION] Retornando las {cantidad} mejores opciones:\n")
        
        for i, sug in enumerate(top_suggestions, 1):
            print(f"  Opcion {i}: {sug['hora']} - {sug['razon']}")
        
        return top_suggestions
    
    def _get_score_explanation(self, score: float, hora: str) -> str:
        """
        Genera explicacion legible del score de optimizacion.
        """
        hora_int = int(hora.split(':')[0])
        
        if score < 0:
            return "Horario optimo - alta productividad y baja carga"
        elif score < 10:
            return "Buen horario - buena productividad"
        elif score < 20:
            return "Horario aceptable - carga moderada"
        else:
            return "Horario suboptimo - alta carga o baja productividad"
    
    def optimize_meeting(self, fecha: str, hora_propuesta: str, duracion_minutos: int) -> Dict:
        """
        AUTO-OPTIMIZACION: Evalua y potencialmente mejora un horario propuesto.
        
        Este es el punto de entrada principal para la optimizacion.
        Retorna el horario propuesto si es optimo, o sugiere alternativas mejores.
        """
        print("\n" + "=" * 70)
        print("[AUTO-OPTIMIZACION] Iniciando analisis de optimizacion")
        print("=" * 70)
        
        # Verificar conflictos
        conflicts = self.check_conflicts(fecha, hora_propuesta, duracion_minutos)
        
        if conflicts:
            print(f"[AUTO-OPTIMIZACION] CONFLICTO DETECTADO con {len(conflicts)} reunion(es)")
            print("[AUTO-OPTIMIZACION] Buscando alternativas optimas...")
            
            alternatives = self.suggest_optimal_times(fecha, duracion_minutos, cantidad=3)
            
            return {
                'status': 'conflicto',
                'hora_original': hora_propuesta,
                'conflictos': conflicts,
                'alternativas': alternatives
            }
        else:
            # Calcular score del horario propuesto
            score = self.calculate_load_score(fecha, hora_propuesta)
            
            # Buscar si hay opciones significativamente mejores
            alternatives = self.suggest_optimal_times(fecha, duracion_minutos, cantidad=3)
            
            best_alternative_score = alternatives[0]['score'] if alternatives else float('inf')
            
            # Si hay una alternativa mucho mejor (diferencia > 15 puntos), sugerirla
            if best_alternative_score < score - 15:
                print(f"[AUTO-OPTIMIZACION] El horario propuesto es valido pero suboptimo")
                print(f"[AUTO-OPTIMIZACION] Se encontraron opciones mejores")
                
                return {
                    'status': 'sugerencia_mejora',
                    'hora_original': hora_propuesta,
                    'score_original': score,
                    'alternativas': alternatives
                }
            else:
                print(f"[AUTO-OPTIMIZACION] Horario propuesto es optimo")
                print(f"[AUTO-OPTIMIZACION] Score: {score}")
                
                return {
                    'status': 'optimo',
                    'hora_aprobada': hora_propuesta,
                    'score': score
                }

# Ejemplo de uso del motor de auto-optimizacion
optimizer = AutoOptimizationEngine()

# Simular agenda con algunas reuniones existentes
optimizer.add_to_agenda({
    'fecha': '2025-12-02',
    'hora': '10:00',
    'duracion': 60,
    'asunto': 'Reunion de equipo'
})
optimizer.add_to_agenda({
    'fecha': '2025-12-02',
    'hora': '14:00',
    'duracion': 30,
    'asunto': 'Seguimiento proyecto X'
})

# Caso 1: Intentar agendar en un horario con conflicto
print("\nCASO 1: Horario con conflicto")
resultado1 = optimizer.optimize_meeting(
    fecha='2025-12-02',
    hora_propuesta='10:30',
    duracion_minutos=45
)

# Caso 2: Intentar agendar en horario suboptimo
print("\n\nCASO 2: Horario valido pero suboptimo")
resultado2 = optimizer.optimize_meeting(
    fecha='2025-12-02',
    hora_propuesta='17:00',
    duracion_minutos=30
)

## Principio 3: AUTO-CURACION (Self-Healing)

La auto-curacion permite que el sistema detecte, diagnostique y repare problemas automaticamente. En el contexto de agendamiento de reuniones:

1. Detecta cuando faltan datos criticos (fecha, hora, asunto, descripcion)
2. Diagnostica que informacion especifica falta
3. Repara solicitando los datos faltantes de forma inteligente
4. Verifica que la reparacion fue exitosa antes de continuar

Este principio ya fue implementado extensivamente en el notebook original. A continuacion se presenta una version simplificada enfocada en los conceptos clave.

In [ ]:
# ========================================================================
# PRINCIPIO 3: AUTO-CURACION (Version simplificada)
# Sistema que detecta, diagnostica y repara problemas automaticamente
# ========================================================================

class AutoHealingEngine:
    """
    Motor de auto-curacion que implementa el ciclo completo:
    Deteccion -> Diagnostico -> Reparacion -> Verificacion
    
    Este componente implementa el tercer principio de computacion autonomica:
    la capacidad del sistema de detectar y corregir problemas automaticamente.
    """
    
    def __init__(self):
        self.required_fields = ['fecha', 'hora', 'asunto', 'descripcion']
    
    def detect(self, data: Dict) -> bool:
        """
        AUTO-CURACION PASO 1: DETECCION
        
        Detecta automaticamente si hay problemas en los datos.
        Retorna True si todo esta correcto, False si hay problemas.
        """
        print("\n[AUTO-CURACION - DETECCION] Analizando integridad de datos...")
        
        for field in self.required_fields:
            if field not in data or not data[field]:
                print(f"[AUTO-CURACION - DETECCION] Problema detectado: falta campo '{field}'")
                return False
        
        print("[AUTO-CURACION - DETECCION] No se detectaron problemas")
        return True
    
    def diagnose(self, data: Dict) -> Dict:
        """
        AUTO-CURACION PASO 2: DIAGNOSTICO
        
        Diagnostica que campos especificos faltan o estan incorrectos.
        Retorna un diagnostico detallado del problema.
        """
        print("\n[AUTO-CURACION - DIAGNOSTICO] Diagnosticando problemas especificos...")
        
        diagnosis = {
            'missing_fields': [],
            'present_fields': {},
            'severity': 'none'
        }
        
        for field in self.required_fields:
            if field not in data or not data[field]:
                diagnosis['missing_fields'].append(field)
                print(f"  - Campo faltante: {field}")
            else:
                diagnosis['present_fields'][field] = data[field]
                print(f"  - Campo presente: {field} = {data[field]}")
        
        # Calcular severidad
        num_missing = len(diagnosis['missing_fields'])
        if num_missing == 0:
            diagnosis['severity'] = 'none'
        elif num_missing <= 2:
            diagnosis['severity'] = 'baja'
        else:
            diagnosis['severity'] = 'alta'
        
        print(f"\n[AUTO-CURACION - DIAGNOSTICO] Severidad: {diagnosis['severity']}")
        print(f"[AUTO-CURACION - DIAGNOSTICO] Campos faltantes: {len(diagnosis['missing_fields'])}")
        
        return diagnosis
    
    def repair(self, diagnosis: Dict, user_input_callback=None) -> Dict:
        """
        AUTO-CURACION PASO 3: REPARACION
        
        Repara los problemas diagnosticados solicitando datos al usuario.
        En un sistema real, podria intentar reparacion automatica primero.
        """
        print("\n[AUTO-CURACION - REPARACION] Iniciando proceso de reparacion...")
        
        repaired_data = diagnosis['present_fields'].copy()
        
        # Para cada campo faltante, solicitarlo (simulado)
        for field in diagnosis['missing_fields']:
            print(f"[AUTO-CURACION - REPARACION] Solicitando campo: {field}")
            
            # En este ejemplo, usar valores predeterminados para demostracion
            # En produccion, esto solicitaria al usuario
            default_values = {
                'fecha': '2025-12-05',
                'hora': '10:00',
                'asunto': 'Reunion por definir',
                'descripcion': 'Descripcion pendiente'
            }
            
            repaired_data[field] = default_values[field]
            print(f"  -> Valor reparado: {field} = {repaired_data[field]}")
        
        print("\n[AUTO-CURACION - REPARACION] Reparacion completada")
        return repaired_data
    
    def verify(self, data: Dict) -> bool:
        """
        AUTO-CURACION PASO 4: VERIFICACION
        
        Verifica que la reparacion fue exitosa.
        Este paso cierra el ciclo de auto-curacion.
        """
        print("\n[AUTO-CURACION - VERIFICACION] Verificando que la reparacion fue exitosa...")
        
        is_complete = self.detect(data)
        
        if is_complete:
            print("[AUTO-CURACION - VERIFICACION] Verificacion exitosa - todos los campos presentes")
            return True
        else:
            print("[AUTO-CURACION - VERIFICACION] Verificacion fallida - aun faltan campos")
            return False
    
    def heal(self, data: Dict) -> Dict:
        """
        AUTO-CURACION: Orquestador del ciclo completo de curacion.
        
        Ejecuta los 4 pasos en orden:
        1. Deteccion
        2. Diagnostico
        3. Reparacion
        4. Verificacion
        """
        print("\n" + "=" * 70)
        print("[AUTO-CURACION] Iniciando ciclo de auto-curacion")
        print("=" * 70)
        
        # Paso 1: Deteccion
        is_healthy = self.detect(data)
        
        if is_healthy:
            print("\n[AUTO-CURACION] Sistema saludable - no requiere reparacion")
            return data
        
        # Paso 2: Diagnostico
        diagnosis = self.diagnose(data)
        
        # Paso 3: Reparacion
        repaired_data = self.repair(diagnosis)
        
        # Paso 4: Verificacion
        is_verified = self.verify(repaired_data)
        
        if is_verified:
            print("\n" + "=" * 70)
            print("[AUTO-CURACION] Ciclo completado exitosamente")
            print("=" * 70)
            return repaired_data
        else:
            raise Exception("[AUTO-CURACION] Error: La verificacion fallo despues de la reparacion")

# Ejemplo de uso del motor de auto-curacion
healer = AutoHealingEngine()

# Caso 1: Datos incompletos que requieren curacion
print("\nCASO: Datos incompletos")
datos_incompletos = {
    'asunto': 'Reunion con el equipo',
    'descripcion': 'Discutir avances del proyecto'
    # Faltan: fecha y hora
}

datos_curados = healer.heal(datos_incompletos)

print("\n\nDATOS FINALES DESPUES DE AUTO-CURACION:")
for key, value in datos_curados.items():
    print(f"  {key}: {value}")

## Principio 4: AUTO-PROTECCION (Self-Protection)

La auto-proteccion permite que el sistema se defienda de ataques maliciosos y usos indebidos. En el contexto de agendamiento de reuniones:

1. Valida todas las entradas del usuario para prevenir inyeccion de codigo
2. Protege informacion sensible como tokens y credenciales
3. Detecta patrones anomalos de uso (rate limiting)
4. Sanitiza datos antes de procesarlos
5. Implementa autenticacion y autorizacion

Este principio es critico para la seguridad del sistema en ambientes de produccion.

In [ ]:
# ========================================================================
# PRINCIPIO 4: AUTO-PROTECCION
# Sistema que se defiende automaticamente de amenazas y usos indebidos
# ========================================================================

class AutoProtectionEngine:
    """
    Motor de auto-proteccion que implementa multiples capas de seguridad
    para proteger el sistema de amenazas y usos indebidos.
    
    Este componente implementa el cuarto principio de computacion autonomica:
    la capacidad del sistema de defenderse automaticamente de amenazas
    sin intervencion manual.
    """
    
    def __init__(self):
        # Rate limiting: rastrear solicitudes por usuario
        self.request_history = {}  # {user_id: [timestamps]}
        self.max_requests_per_minute = 10
        self.max_requests_per_hour = 60
        
        # Patrones maliciosos a detectar
        self.malicious_patterns = [
            r'<script[^>]*>',  # XSS
            r'javascript:',     # XSS
            r'DROP\s+TABLE',   # SQL Injection
            r'--',             # SQL comments
            r'UNION\s+SELECT', # SQL Injection
            r'eval\s*\(',      # Code injection
            r'exec\s*\(',      # Code injection
            r'__import__',     # Python injection
        ]
        
        # Lista negra de usuarios bloqueados
        self.blocked_users = set()
    
    def sanitize_input(self, text: str) -> str:
        """
        AUTO-PROTECCION: Sanitiza entradas del usuario para prevenir inyecciones.
        
        Este metodo es la primera linea de defensa contra ataques.
        Remueve o escapa caracteres peligrosos.
        """
        if not isinstance(text, str):
            return str(text)
        
        # Remover caracteres de control peligrosos
        sanitized = text.replace('\x00', '')  # Null bytes
        
        # Escapar caracteres HTML peligrosos
        html_escape_table = {
            '<': '&lt;',
            '>': '&gt;',
            '"': '&quot;',
            "'": '&#x27;',
        }
        
        for char, escaped in html_escape_table.items():
            sanitized = sanitized.replace(char, escaped)
        
        return sanitized
    
    def detect_malicious_input(self, text: str) -> tuple[bool, List[str]]:
        """
        AUTO-PROTECCION: Detecta patrones maliciosos en las entradas.
        
        Analiza el texto en busca de patrones conocidos de ataques.
        Retorna (es_malicioso, lista_de_patrones_detectados).
        """
        print("\n[AUTO-PROTECCION] Analizando entrada en busca de patrones maliciosos...")
        
        detected_patterns = []
        
        for pattern in self.malicious_patterns:
            if re.search(pattern, text, re.IGNORECASE):
                detected_patterns.append(pattern)
        
        if detected_patterns:
            print(f"[AUTO-PROTECCION] ALERTA: Se detectaron {len(detected_patterns)} patron(es) malicioso(s)")
            return True, detected_patterns
        else:
            print("[AUTO-PROTECCION] No se detectaron patrones maliciosos")
            return False, []
    
    def check_rate_limit(self, user_id: str) -> tuple[bool, str]:
        """
        AUTO-PROTECCION: Implementa rate limiting para prevenir abuso.
        
        Verifica si el usuario ha excedido los limites de solicitudes.
        Retorna (permitido, mensaje).
        """
        print(f"\n[AUTO-PROTECCION] Verificando rate limit para usuario: {user_id}")
        
        current_time = time.time()
        
        # Inicializar historial si no existe
        if user_id not in self.request_history:
            self.request_history[user_id] = []
        
        # Limpiar solicitudes antiguas (mas de 1 hora)
        self.request_history[user_id] = [
            timestamp for timestamp in self.request_history[user_id]
            if current_time - timestamp < 3600  # 1 hora
        ]
        
        # Contar solicitudes recientes
        requests_last_minute = sum(
            1 for timestamp in self.request_history[user_id]
            if current_time - timestamp < 60
        )
        
        requests_last_hour = len(self.request_history[user_id])
        
        # Verificar limites
        if requests_last_minute >= self.max_requests_per_minute:
            print(f"[AUTO-PROTECCION] BLOQUEADO: Excedio limite por minuto ({requests_last_minute}/{self.max_requests_per_minute})")
            return False, f"Demasiadas solicitudes. Limite: {self.max_requests_per_minute}/minuto"
        
        if requests_last_hour >= self.max_requests_per_hour:
            print(f"[AUTO-PROTECCION] BLOQUEADO: Excedio limite por hora ({requests_last_hour}/{self.max_requests_per_hour})")
            return False, f"Demasiadas solicitudes. Limite: {self.max_requests_per_hour}/hora"
        
        # Registrar esta solicitud
        self.request_history[user_id].append(current_time)
        
        print(f"[AUTO-PROTECCION] Rate limit OK - Solicitudes: {requests_last_minute}/min, {requests_last_hour}/hora")
        return True, "OK"
    
    def validate_meeting_data(self, data: Dict) -> tuple[bool, List[str]]:
        """
        AUTO-PROTECCION: Valida que los datos de la reunion sean seguros y validos.
        
        Implementa validaciones de negocio para prevenir datos malformados.
        Retorna (es_valido, lista_de_errores).
        """
        print("\n[AUTO-PROTECCION] Validando datos de reunion...")
        
        errors = []
        
        # Validar fecha
        if 'fecha' in data:
            try:
                fecha_dt = datetime.strptime(data['fecha'], '%Y-%m-%d')
                
                # No permitir fechas en el pasado
                if fecha_dt.date() < datetime.now().date():
                    errors.append("Fecha no puede estar en el pasado")
                
                # No permitir fechas muy lejanas (mas de 1 ano)
                if fecha_dt.date() > (datetime.now() + timedelta(days=365)).date():
                    errors.append("Fecha no puede estar mas de 1 ano en el futuro")
                    
            except ValueError:
                errors.append("Formato de fecha invalido (debe ser YYYY-MM-DD)")
        
        # Validar hora
        if 'hora' in data:
            try:
                hora_dt = datetime.strptime(data['hora'], '%H:%M')
                
                # Validar horario laboral razonable
                if hora_dt.hour < 6 or hora_dt.hour > 22:
                    errors.append("Hora fuera del rango razonable (06:00 - 22:00)")
                    
            except ValueError:
                errors.append("Formato de hora invalido (debe ser HH:MM)")
        
        # Validar longitud de asunto
        if 'asunto' in data:
            if len(data['asunto']) > 200:
                errors.append("Asunto demasiado largo (maximo 200 caracteres)")
            if len(data['asunto']) < 3:
                errors.append("Asunto demasiado corto (minimo 3 caracteres)")
        
        # Validar longitud de descripcion
        if 'descripcion' in data:
            if len(data['descripcion']) > 1000:
                errors.append("Descripcion demasiado larga (maximo 1000 caracteres)")
        
        if errors:
            print(f"[AUTO-PROTECCION] Validacion fallida - {len(errors)} error(es) encontrado(s)")
            for error in errors:
                print(f"  - {error}")
            return False, errors
        else:
            print("[AUTO-PROTECCION] Validacion exitosa")
            return True, []
    
    def protect(self, user_id: str, data: Dict) -> Dict:
        """
        AUTO-PROTECCION: Punto de entrada principal del motor de proteccion.
        
        Ejecuta todas las verificaciones de seguridad:
        1. Rate limiting
        2. Deteccion de patrones maliciosos
        3. Sanitizacion de entradas
        4. Validacion de datos
        
        Retorna datos sanitizados si pasan todas las verificaciones,
        o lanza excepcion si se detecta amenaza.
        """
        print("\n" + "=" * 70)
        print("[AUTO-PROTECCION] Iniciando verificaciones de seguridad")
        print("=" * 70)
        
        # Verificacion 1: Usuario bloqueado
        if user_id in self.blocked_users:
            raise Exception(f"[AUTO-PROTECCION] Usuario {user_id} esta bloqueado")
        
        # Verificacion 2: Rate limiting
        allowed, message = self.check_rate_limit(user_id)
        if not allowed:
            raise Exception(f"[AUTO-PROTECCION] Rate limit excedido: {message}")
        
        # Verificacion 3: Deteccion de patrones maliciosos
        for field, value in data.items():
            if isinstance(value, str):
                is_malicious, patterns = self.detect_malicious_input(value)
                if is_malicious:
                    # Bloquear usuario si se detecta comportamiento malicioso
                    self.blocked_users.add(user_id)
                    raise Exception(f"[AUTO-PROTECCION] Patron malicioso detectado en '{field}'. Usuario bloqueado.")
        
        # Verificacion 4: Sanitizacion
        print("\n[AUTO-PROTECCION] Sanitizando entradas...")
        sanitized_data = {}
        for field, value in data.items():
            if isinstance(value, str):
                sanitized_data[field] = self.sanitize_input(value)
            else:
                sanitized_data[field] = value
        print("[AUTO-PROTECCION] Sanitizacion completada")
        
        # Verificacion 5: Validacion de datos
        is_valid, errors = self.validate_meeting_data(sanitized_data)
        if not is_valid:
            raise Exception(f"[AUTO-PROTECCION] Validacion fallida: {', '.join(errors)}")
        
        print("\n" + "=" * 70)
        print("[AUTO-PROTECCION] Todas las verificaciones pasaron exitosamente")
        print("=" * 70)
        
        return sanitized_data

# Ejemplo de uso del motor de auto-proteccion
protector = AutoProtectionEngine()

# Caso 1: Entrada legitima
print("\nCASO 1: Entrada legitima")
try:
    datos_seguros = protector.protect(
        user_id="user123",
        data={
            'fecha': '2025-12-10',
            'hora': '14:00',
            'asunto': 'Reunion de planificacion',
            'descripcion': 'Discutir objetivos del proximo trimestre'
        }
    )
    print("\nDATOS PROTEGIDOS Y VALIDADOS:")
    for key, value in datos_seguros.items():
        print(f"  {key}: {value}")
except Exception as e:
    print(f"\nERROR: {e}")

# Caso 2: Intento de inyeccion de codigo
print("\n\nCASO 2: Intento de inyeccion de codigo (XSS)")
try:
    datos_maliciosos = protector.protect(
        user_id="attacker456",
        data={
            'fecha': '2025-12-10',
            'hora': '14:00',
            'asunto': '<script>alert("XSS")</script>',
            'descripcion': 'Intento de ataque'
        }
    )
except Exception as e:
    print(f"\nERROR (esperado): {e}")

# Caso 3: Fecha invalida
print("\n\nCASO 3: Fecha en el pasado (invalida)")
try:
    datos_invalidos = protector.protect(
        user_id="user789",
        data={
            'fecha': '2020-01-01',
            'hora': '14:00',
            'asunto': 'Reunion pasada',
            'descripcion': 'Esto deberia fallar'
        }
    )
except Exception as e:
    print(f"\nERROR (esperado): {e}")

## Sistema Integrado: Los 4 Principios Trabajando Juntos

A continuacion se presenta un ejemplo de como los cuatro principios de la computacion autonomica trabajan juntos en un sistema integrado de agendamiento de reuniones.

El flujo completo es:
1. AUTO-PROTECCION valida y sanitiza las entradas
2. AUTO-CURACION detecta y repara datos faltantes
3. AUTO-CONFIGURACION configura parametros automaticamente
4. AUTO-OPTIMIZACION sugiere los mejores horarios

Este enfoque integrado demuestra el verdadero poder de la computacion autonomica.

In [ ]:
# ========================================================================
# SISTEMA INTEGRADO: Orquestador de los 4 Principios Autonomicos
# ========================================================================

class AutonomicMeetingScheduler:
    """
    Sistema integrado que orquesta los 4 principios de computacion autonomica
    para crear un asistente de agendamiento completamente autonomo.
    
    Este sistema demuestra como los principios trabajan juntos para crear
    un sistema verdaderamente auto-gestionado que requiere minima intervencion humana.
    """
    
    def __init__(self):
        # Inicializar los 4 motores autonomicos
        self.protector = AutoProtectionEngine()      # Principio 4
        self.healer = AutoHealingEngine()            # Principio 3
        self.configurator = AutoConfigurationEngine() # Principio 1
        self.optimizer = AutoOptimizationEngine()     # Principio 2
    
    def schedule_meeting(self, user_id: str, raw_data: Dict) -> Dict:
        """
        Agenda una reunion aplicando los 4 principios autonomicos en secuencia.
        
        Flujo autonomico completo:
        1. AUTO-PROTECCION: Validar y proteger contra amenazas
        2. AUTO-CURACION: Detectar y reparar datos faltantes
        3. AUTO-CONFIGURACION: Configurar parametros automaticamente
        4. AUTO-OPTIMIZACION: Optimizar horarios
        """
        print("\n" + "=" * 70)
        print("SISTEMA AUTONOMICO DE AGENDAMIENTO DE REUNIONES")
        print("Aplicando los 4 principios de computacion autonomica")
        print("=" * 70)
        
        # PRINCIPIO 4: AUTO-PROTECCION
        # Primera linea de defensa - proteger el sistema
        print("\n[PASO 1/4] Aplicando AUTO-PROTECCION...")
        try:
            protected_data = self.protector.protect(user_id, raw_data)
        except Exception as e:
            print(f"\nERROR DE SEGURIDAD: {e}")
            return {'status': 'rechazado', 'razon': str(e)}
        
        # PRINCIPIO 3: AUTO-CURACION
        # Asegurar que los datos esten completos
        print("\n[PASO 2/4] Aplicando AUTO-CURACION...")
        healed_data = self.healer.heal(protected_data)
        
        # PRINCIPIO 1: AUTO-CONFIGURACION
        # Configurar parametros automaticamente
        print("\n[PASO 3/4] Aplicando AUTO-CONFIGURACION...")
        config = self.configurator.auto_configure(
            asunto=healed_data['asunto'],
            descripcion=healed_data['descripcion']
        )
        
        # PRINCIPIO 2: AUTO-OPTIMIZACION
        # Optimizar horarios
        print("\n[PASO 4/4] Aplicando AUTO-OPTIMIZACION...")
        optimization_result = self.optimizer.optimize_meeting(
            fecha=healed_data['fecha'],
            hora_propuesta=healed_data['hora'],
            duracion_minutos=config['duracion_minutos']
        )
        
        # Integrar resultados
        final_meeting = {
            'fecha': healed_data['fecha'],
            'hora': healed_data['hora'],
            'asunto': healed_data['asunto'],
            'descripcion': healed_data['descripcion'],
            'configuracion': config,
            'optimizacion': optimization_result,
            'status': 'agendado'
        }
        
        print("\n" + "=" * 70)
        print("REUNION AGENDADA EXITOSAMENTE")
        print("Sistema autonomico completo todos los pasos")
        print("=" * 70)
        
        return final_meeting

# Demostrar el sistema integrado
scheduler = AutonomicMeetingScheduler()

print("\n" + "#" * 70)
print("DEMOSTRACION: Sistema Autonomico Integrado")
print("#" * 70)

# Datos de entrada parciales (para demostrar auto-curacion)
entrada_usuario = {
    'asunto': 'Revision de estrategia Q4',
    'descripcion': 'Planificar objetivos estrategicos para el ultimo trimestre del ano'
    # Faltan: fecha y hora (seran curados)
}

# Agendar reunion usando el sistema autonomico completo
reunion_final = scheduler.schedule_meeting(
    user_id="demo_user_001",
    raw_data=entrada_usuario
)

# Mostrar resultado final
print("\n\n" + "=" * 70)
print("RESULTADO FINAL")
print("=" * 70)
print(f"\nFecha: {reunion_final['fecha']}")
print(f"Hora: {reunion_final['hora']}")
print(f"Asunto: {reunion_final['asunto']}")
print(f"Descripcion: {reunion_final['descripcion']}")
print(f"\nTipo de reunion: {reunion_final['configuracion']['tipo']}")
print(f"Duracion: {reunion_final['configuracion']['duracion_minutos']} minutos")
print(f"Prioridad: {reunion_final['configuracion']['prioridad']}")
print(f"Formato: {reunion_final['configuracion']['formato']}")
print(f"\nEstado de optimizacion: {reunion_final['optimizacion']['status']}")
print("\n" + "=" * 70)

## Conclusiones

Este notebook ha demostrado como implementar los cuatro principios fundamentales de la computacion autonomica en el contexto de un sistema de agendamiento de reuniones:

### 1. Auto-configuracion (Self-Configuration)
- El sistema detecta automaticamente el tipo de reunion basandose en palabras clave
- Configura duracion, prioridad y recordatorios sin intervencion manual
- Se adapta al contexto proporcionado por el usuario

### 2. Auto-optimizacion (Self-Optimization)
- Detecta conflictos en horarios automaticamente
- Sugiere los mejores horarios basandose en patrones de productividad
- Balancea la carga de reuniones para evitar sobrecarga
- Mejora continuamente las sugerencias

### 3. Auto-curacion (Self-Healing)
- Detecta cuando faltan datos criticos
- Diagnostica que informacion especifica falta
- Repara solicitando datos o usando valores inteligentes
- Verifica que la reparacion fue exitosa

### 4. Auto-proteccion (Self-Protection)
- Valida todas las entradas del usuario
- Detecta y bloquea patrones maliciosos
- Implementa rate limiting para prevenir abuso
- Sanitiza datos antes de procesarlos
- Protege el sistema de amenazas automaticamente

### Beneficios del Enfoque Autonomico

1. **Reduccion de carga cognitiva**: El usuario solo necesita proporcionar informacion minima
2. **Mayor seguridad**: El sistema se protege automaticamente de amenazas
3. **Mejor experiencia**: Sugerencias inteligentes y configuracion automatica
4. **Resiliencia**: El sistema detecta y repara problemas sin intervencion
5. **Eficiencia**: Optimizacion continua del uso de recursos (tiempo, energia)

La computacion autonomica representa el futuro de los sistemas inteligentes, donde la tecnologia se adapta y gestiona a si misma de manera similar al sistema nervioso autonomo humano.